# 1. Mount Google Drive & Install Dependencies

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Install PyTorch with CUDA 12.1
!pip install torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121 -q
!pip install torchvision==0.18.1 -q
!pip install transformers==4.40.0 -q
!pip install tokenizers==0.19.1 -q
!pip install torchmetrics==1.4.0 -q
!pip install medpy -q
!pip install timm einops albumentations -q

In [ ]:
#@title Install mamba_ssm and causal-conv1d
!pip install causal-conv1d==1.4.0 -q
!pip install mamba-ssm==2.2.2 -q

In [ ]:
import torch
print(torch.version.cuda)
print(torch.__version__)

# 2. Copy Custom Modules from Drive
Assume your custom vmamba.py and rmsnet.py are in /content/lib (adjust path as needed).

In [ ]:
#@title Copy VMamba and RMSNet modules
import shutil
import os

drive_source = '/content/lib'   # <-- change to your folder
for file in ['vmamba.py', 'rmsnet.py']:
    src = os.path.join(drive_source, file)
    if os.path.exists(src):
        shutil.copy(src, '.')
        print(f"✅ {file} loaded")
    else:
        print(f"❌ Missing: {file}")

# Fix relative import in rmsnet.py
with open('rmsnet.py', 'r') as f:
    content = f.read()
content = content.replace('from .vmamba import VSSM', 'from vmamba import VSSM')
with open('rmsnet.py', 'w') as f:
    f.write(content)
print("Fixed rmsnet.py import")

from rmsnet import RMSNet
print("RMSNet imported successfully!")

# 3. Import All Necessary Libraries

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torchmetrics
from torchmetrics import JaccardIndex, F1Score, Accuracy, Precision, Recall, Specificity
from medpy.metric import binary

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 4. Dataset Paths and Configuration

In [ ]:
#@title Set Data Directories
data_dir = "/content/drive/MyDrive/breast_density_data"   # <-- change to your dataset root
image_dir = os.path.join(data_dir, "images")
breast_mask_dir = os.path.join(data_dir, "breast_masks")
dense_mask_dir = os.path.join(data_dir, "dense_masks")

IMG_HEIGHT = 256
IMG_WIDTH = 256
batch_size = 8
learning_rate = 1e-3
num_epochs = 100
patience = 40

os.makedirs("models", exist_ok=True)
os.makedirs("logs", exist_ok=True)
os.makedirs("results", exist_ok=True)

#  5. Dataset Class for Breast Density (Multi-Label with 2 Masks)

In [ ]:
class BreastDensityDataset(Dataset):
    def __init__(self, image_paths, breast_paths, dense_paths, transform=None):
        self.image_paths = image_paths
        self.breast_paths = breast_paths
        self.dense_paths = dense_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Read image (RGB)
        image = cv2.imread(self.image_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Read breast mask (grayscale, 0/255) -> binary 0/1
        breast = cv2.imread(self.breast_paths[idx], cv2.IMREAD_GRAYSCALE)
        breast = (breast > 0).astype(np.float32)

        # Read dense mask (grayscale, 0/255) -> binary 0/1
        dense = cv2.imread(self.dense_paths[idx], cv2.IMREAD_GRAYSCALE)
        dense = (dense > 0).astype(np.float32)

        # Stack masks into two-channel target: shape (H, W, 2)
        mask = np.stack([breast, dense], axis=-1)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']              # (H, W, 2) from ToTensorV2
            mask = mask.permute(2, 0, 1)          # (2, H, W)
        else:
            image = torch.from_numpy(image).permute(2,0,1).float() / 255.0
            mask = torch.from_numpy(mask).permute(2,0,1).float()  # (2, H, W)

        return image, mask

# 6. Transforms (Albumentations)

In [ ]:
# For RGB images (ImageNet normalization)
train_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# 7. Load Data and Split

In [ ]:
# Get file lists (assuming same basenames)
image_files = sorted(os.listdir(image_dir))
breast_files = sorted(os.listdir(breast_mask_dir))
dense_files = sorted(os.listdir(dense_mask_dir))

assert len(image_files) == len(breast_files) == len(dense_files)

image_paths = [os.path.join(image_dir, f) for f in image_files]
breast_paths = [os.path.join(breast_mask_dir, f) for f in breast_files]
dense_paths = [os.path.join(dense_mask_dir, f) for f in dense_files]

# Split into train/val/test (e.g., 70/15/15)
from sklearn.model_selection import train_test_split
temp = list(zip(image_paths, breast_paths, dense_paths))
train_val, test = train_test_split(temp, test_size=0.15, random_state=42)
train, val = train_test_split(train_val, test_size=0.15/0.85, random_state=42)

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

# 8. Create Datasets and DataLoaders

In [ ]:
train_dataset = BreastDensityDataset(
    [t[0] for t in train], [t[1] for t in train], [t[2] for t in train],
    transform=train_transform
)
val_dataset = BreastDensityDataset(
    [t[0] for t in val], [t[1] for t in val], [t[2] for t in val],
    transform=val_transform
)
test_dataset = BreastDensityDataset(
    [t[0] for t in test], [t[1] for t in test], [t[2] for t in test],
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

# 9. Loss Function (Multi-Label Dice + BCE)

In [ ]:
class MultiLabelDiceBCELoss(nn.Module):
    def __init__(self, dice_weight=0.5, bce_weight=0.5):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()   # expects logits

    def forward(self, logits, targets):
        # logits: (N,2,H,W) raw
        # targets: (N,2,H,W) binary (0/1)
        targets = targets.float()
        bce = self.bce(logits, targets)

        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(2,3))
        union = probs.sum(dim=(2,3)) + targets.sum(dim=(2,3))
        dice = (2. * intersection + 1e-6) / (union + 1e-6)
        dice_loss = 1 - dice.mean()   # average over batch and channels

        return self.dice_weight * dice_loss + self.bce_weight * bce

criterion = MultiLabelDiceBCELoss().to(device)

# 10. Model Initialization (RMSNet with 2 Output Classes)

In [ ]:
from rmsnet import RMSNet

model = RMSNet(
    input_channels=3,
    num_classes=2,               # two masks: breast and dense
    depths=[2, 2, 9, 2],
    depths_decoder=[2, 9, 2, 2],
    drop_path_rate=0.2,
    load_ckpt_path=None
).to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

print(f"Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


# 11. Metrics Setup (Per-Channel)
We compute metrics for each channel separately and report them.

In [ ]:
# We'll define a function to compute per-channel metrics
def compute_per_channel_metrics(preds, targets, threshold=0.5):
    """
    preds: (N,2,H,W) logits
    targets: (N,2,H,W) binary
    Returns a dict with keys like 'breast_dice', 'dense_dice', etc.
    """
    probs = torch.sigmoid(preds)
    pred_bin = (probs > threshold).long()
    metrics = {}
    for i, name in enumerate(['breast', 'dense']):
        p = pred_bin[:, i, :, :]
        t = targets[:, i, :, :]
        # Jaccard
        j = JaccardIndex(task='binary').to(device)
        jaccard = j(p, t).item()
        d = F1Score(task='binary').to(device)
        dice = d(p, t).item()
        acc = Accuracy(task='binary').to(device)
        accuracy = acc(p, t).item()
        prec = Precision(task='binary').to(device)
        precision = prec(p, t).item()
        rec = Recall(task='binary').to(device)
        recall = rec(p, t).item()
        spec = Specificity(task='binary').to(device)
        specificity = spec(p, t).item()
        metrics[f'{name}_jaccard'] = jaccard
        metrics[f'{name}_dice'] = dice
        metrics[f'{name}_accuracy'] = accuracy
        metrics[f'{name}_precision'] = precision
        metrics[f'{name}_recall'] = recall
        metrics[f'{name}_specificity'] = specificity
    return metrics

def compute_hausdorff(pred_mask, true_mask, spacing=(1,1)):
    pred_binary = pred_mask.astype(np.uint8)
    true_binary = true_mask.astype(np.uint8)
    if np.sum(pred_binary) == 0 or np.sum(true_binary) == 0:
        return np.nan, np.nan
    hd = binary.hd(pred_binary, true_binary, voxelspacing=spacing)
    hd95 = binary.hd95(pred_binary, true_binary, voxelspacing=spacing)
    return hd, hd95

# 12. Training and Validation Functions

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for images, masks in tqdm(dataloader, desc='Training'):
        images = images.to(device)
        masks = masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)          # (N,2,H,W) logits
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_metrics = defaultdict(float)
    hd_list_breast, hd95_list_breast = [], []
    hd_list_dense, hd95_list_dense = [], []

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc='Validation'):
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

            # Compute per-channel metrics
            batch_metrics = compute_per_channel_metrics(outputs, masks)
            for k, v in batch_metrics.items():
                all_metrics[k] += v

            # Hausdorff per image per channel
            probs = torch.sigmoid(outputs)
            pred_bin = (probs > 0.5).long()
            for i in range(images.size(0)):
                for ch, name in enumerate(['breast', 'dense']):
                    pred_np = pred_bin[i, ch, :, :].cpu().numpy()
                    mask_np = masks[i, ch, :, :].cpu().numpy()
                    hd, hd95 = compute_hausdorff(pred_np, mask_np)
                    if name == 'breast':
                        hd_list_breast.append(hd); hd95_list_breast.append(hd95)
                    else:
                        hd_list_dense.append(hd); hd95_list_dense.append(hd95)

    avg_loss = total_loss / len(dataloader)
    # average metrics over batches
    for k in all_metrics:
        all_metrics[k] /= len(dataloader)
    all_metrics['loss'] = avg_loss
    all_metrics['breast_hd_mean'] = np.nanmean(hd_list_breast)
    all_metrics['breast_hd95_mean'] = np.nanmean(hd95_list_breast)
    all_metrics['dense_hd_mean'] = np.nanmean(hd_list_dense)
    all_metrics['dense_hd95_mean'] = np.nanmean(hd95_list_dense)
    return dict(all_metrics)

# 13. Training Loop with CSV Logging

In [ ]:
log_file = "logs/training_log.csv"
csv_columns = ['epoch', 'train_loss', 'val_loss'] + \
              [f'{name}_{metric}' for name in ['breast','dense'] for metric in ['jaccard','dice','accuracy','precision','recall','specificity']] + \
              ['breast_hd_mean', 'breast_hd95_mean', 'dense_hd_mean', 'dense_hd95_mean']

if not os.path.exists(log_file):
    pd.DataFrame(columns=csv_columns).to_csv(log_file, index=False)

best_val_loss = float('inf')
patience_counter = 0
history = defaultdict(list)

for epoch in range(1, num_epochs+1):
    print(f"\nEpoch {epoch}/{num_epochs}")
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = validate_one_epoch(model, val_loader, criterion, device)
    val_loss = val_metrics['loss']
    scheduler.step(val_loss)

    # Store history
    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    for k, v in val_metrics.items():
        history[k].append(v)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    for name in ['breast','dense']:
        print(f"{name}: Dice={val_metrics[f'{name}_dice']:.4f}, Jaccard={val_metrics[f'{name}_jaccard']:.4f}")

    # Save to CSV
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss}
    for k in csv_columns:
        if k not in ['epoch', 'train_loss', 'val_loss']:
            row[k] = val_metrics.get(k, np.nan)
    pd.DataFrame([row]).to_csv(log_file, mode='a', header=False, index=False)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'models/best_rmsnet.pth')
        print("Best model saved!")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

torch.save(model.state_dict(), 'models/final_rmsnet.pth')

# 14. Plotting Training Curves

In [ ]:
def plot_individual_curves(history):
    os.makedirs("results", exist_ok=True)
    epochs = history['epoch']

    # Loss
    plt.figure(figsize=(8,6))
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss Curves')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('results/loss_curve.png', dpi=150)
    plt.show()
    print("Saved: loss_curve.png")

    # Dice for both channels
    plt.figure(figsize=(8,6))
    plt.plot(epochs, history['breast_dice'], label='Breast Dice')
    plt.plot(epochs, history['dense_dice'], label='Dense Dice')
    plt.xlabel('Epoch')
    plt.ylabel('Dice')
    plt.title('Dice Curves')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('results/dice_curves.png', dpi=150)
    plt.show()
    print("Saved: dice_curves.png")

    # Jaccard (IoU)
    plt.figure(figsize=(8,6))
    plt.plot(epochs, history['breast_jaccard'], label='Breast IoU')
    plt.plot(epochs, history['dense_jaccard'], label='Dense IoU')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.title('IoU Curves')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('results/iou_curves.png', dpi=150)
    plt.show()
    print("Saved: iou_curves.png")

    # HD95
    plt.figure(figsize=(8,6))
    plt.plot(epochs, history['breast_hd95_mean'], label='Breast HD95')
    plt.plot(epochs, history['dense_hd95_mean'], label='Dense HD95')
    plt.xlabel('Epoch')
    plt.ylabel('HD95 (pixels)')
    plt.title('HD95 Curves')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('results/hd95_curves.png', dpi=150)
    plt.show()
    print("Saved: hd95_curves.png")

plot_individual_curves(history)

# 15. Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('models/best_rmsnet.pth'))
model.eval()

test_loss_total = 0
all_test_metrics = defaultdict(float)
hd_list_breast, hd95_list_breast = [], []
hd_list_dense, hd95_list_dense = [], []

with torch.no_grad():
    for images, masks in tqdm(test_loader, desc='Testing'):
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)
        test_loss_total += loss.item()

        batch_metrics = compute_per_channel_metrics(outputs, masks)
        for k, v in batch_metrics.items():
            all_test_metrics[k] += v

        probs = torch.sigmoid(outputs)
        pred_bin = (probs > 0.5).long()
        for i in range(images.size(0)):
            for ch, name in enumerate(['breast', 'dense']):
                pred_np = pred_bin[i, ch, :, :].cpu().numpy()
                mask_np = masks[i, ch, :, :].cpu().numpy()
                hd, hd95 = compute_hausdorff(pred_np, mask_np)
                if name == 'breast':
                    hd_list_breast.append(hd); hd95_list_breast.append(hd95)
                else:
                    hd_list_dense.append(hd); hd95_list_dense.append(hd95)

avg_test_loss = test_loss_total / len(test_loader)
for k in all_test_metrics:
    all_test_metrics[k] /= len(test_loader)
all_test_metrics['loss'] = avg_test_loss
all_test_metrics['breast_hd_mean'] = np.nanmean(hd_list_breast)
all_test_metrics['breast_hd95_mean'] = np.nanmean(hd95_list_breast)
all_test_metrics['dense_hd_mean'] = np.nanmean(hd_list_dense)
all_test_metrics['dense_hd95_mean'] = np.nanmean(hd95_list_dense)

print("\n===== Test Set Results =====")
print(f"Test Loss: {avg_test_loss:.4f}")
for name in ['breast','dense']:
    print(f"{name}: Dice={all_test_metrics[f'{name}_dice']:.4f}, IoU={all_test_metrics[f'{name}_jaccard']:.4f}, HD95={all_test_metrics[f'{name}_hd95_mean']:.2f}")

# 16. Visualise Test Predictions (Breast and Dense)

In [ ]:
def visualize_test_predictions(model, dataset, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples*4))
    indices = random.sample(range(len(dataset)), num_samples)

    for row, idx in enumerate(indices):
        img, mask = dataset[idx]
        img_tensor = img.unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(img_tensor)   # (1,2,H,W)
            probs = torch.sigmoid(output).cpu().numpy()[0]
            breast_pred = (probs[0] > 0.5).astype(np.uint8)
            dense_pred = (probs[1] > 0.5).astype(np.uint8)

        # Denormalize image
        img_np = img.permute(1,2,0).cpu().numpy()
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_np = img_np * std + mean
        img_np = np.clip(img_np, 0, 1)

        mask_np = mask.cpu().numpy()  # (2,H,W)
        breast_gt = mask_np[0]
        dense_gt = mask_np[1]

        axes[row,0].imshow(img_np)
        axes[row,0].set_title('Image')
        axes[row,0].axis('off')

        axes[row,1].imshow(breast_gt, cmap='gray')
        axes[row,1].set_title('Breast GT')
        axes[row,1].axis('off')

        axes[row,2].imshow(dense_gt, cmap='gray')
        axes[row,2].set_title('Dense GT')
        axes[row,2].axis('off')

        # Overlay: breast in red, dense in green
        overlay = (img_np * 255).astype(np.uint8).copy()
        overlay[breast_pred > 0, 0] = 200
        overlay[dense_pred > 0, 1] = 200
        axes[row,3].imshow(overlay)
        axes[row,3].set_title('Pred (breast=red, dense=green)')
        axes[row,3].axis('off')

    plt.tight_layout()
    plt.savefig('results/test_predictions.png', dpi=150)
    plt.show()

visualize_test_predictions(model, test_dataset, num_samples=3)

# 17. Save Final Results

In [ ]:
# Save test metrics to CSV
test_results = pd.DataFrame([{
    'loss': avg_test_loss,
    'breast_dice': all_test_metrics['breast_dice'],
    'breast_iou': all_test_metrics['breast_jaccard'],
    'breast_hd95': all_test_metrics['breast_hd95_mean'],
    'dense_dice': all_test_metrics['dense_dice'],
    'dense_iou': all_test_metrics['dense_jaccard'],
    'dense_hd95': all_test_metrics['dense_hd95_mean']
}])
test_results.to_csv('results/test_metrics.csv', index=False)
print("Test metrics saved to results/test_metrics.csv")